# BERT on WikiText — encoder activations in memory

Every transformer block outputs `(batch, seq_len, hidden)`, so a text run costs
far more per sample than a vision one. Downloads on first run: WikiText-2
(~5 MB) and BERT weights (~440 MB).

In [ ]:
import sys
from pathlib import Path

# Make the repo-local examples._utils package importable when this notebook
# is opened directly from examples/text/, without installing anything extra.
sys.path.insert(0, str(Path.cwd().parents[1]))

import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

from examples._utils.data import activation_loader
from examples._utils.text import WikiTextSamples
from nnact import ActivationPipeline
from nnact._model._hooked import HookedModel

MODEL = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
# A token-classification head gives per-token logits; its weights are
# randomly initialized on top of pretrained BERT, so predictions are
# meaningless here — only the encoder activations are of interest.
model = AutoModelForTokenClassification.from_pretrained(MODEL)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
dataset = WikiTextSamples(tokenizer, n=256, max_length=64)
print(f"{len(dataset)} passages | input_ids {tuple(dataset[0]['input_ids'].shape)}")
print(f"real tokens in first: {int(dataset[0]['attention_mask'].sum())}")
print(dataset.texts[0][:90], "...")


In [ ]:
hooked = HookedModel(model)

# depth=3 reaches the individual blocks; depth=1 would only show
# "embeddings" and "encoder".
hooked.summary(depth=3).head(12)

In [ ]:
# Embeddings, an early block, a middle block, and the last one.
LAYERS = [
    "bert.embeddings",
    "bert.encoder.layer.0",
    "bert.encoder.layer.5",
    "bert.encoder.layer.11",
]

# run() accumulates every batch's activations in memory and hands back an
# ActivationDataset once the run finishes.
pipeline = ActivationPipeline(model, LAYERS, output_type="token")
loader = activation_loader(dataset, batch_size=32)
activations = pipeline.run(loader)
activations.summary()

In [ ]:
# Token-shaped activations are flattened across the batch: one row per real
# token, padding dropped, offsets marking where each passage's tokens start.
last = activations.activations["bert.encoder.layer.11"]
print("layer 11 flattened:", tuple(last.shape))

# The first passage's own tokens, sliced out via the accumulated offsets.
first = activations[0]
first_layer = first.activations["bert.encoder.layer.11"]
print("first passage:", tuple(first_layer.shape))

# One vector per passage: mean-pool each passage's own tokens.
pooled = torch.stack(
    [
        activations[i].activations["bert.encoder.layer.11"].mean(dim=0)
        for i in range(len(activations))
    ]
)
print("mean-pooled:", tuple(pooled.shape))

In [ ]:
# How far each layer's representation sits from the final one.
for name in activations.layer_names:
    vecs = torch.stack(
        [activations[i].activations[name].mean(dim=0) for i in range(len(activations))]
    )
    sim = torch.nn.functional.cosine_similarity(vecs, pooled, dim=1).mean()
    print(f"  {name:<24} cos(layer, final) = {sim:.3f}")